In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_5393_Chandni_Chowk_Delhi_IITM_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,155.89,264.25,20.76,41.03,38.75,46.73,28.00,2.12,25.22,...,1.82,30.14,29.52,NaN,NaN,0.0,0.0,NaN,996.5,NaN
362,2024-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,64.18,129.35,39.99,66.56,68.00,67.10,23.06,2.22,32.59,...,1.16,30.00,29.56,NaN,NaN,0.0,0.0,NaN,996.5,NaN
364,2024-12-30,59.47,135.68,49.97,59.56,71.49,75.20,20.70,1.70,27.04,...,1.54,24.11,29.56,NaN,NaN,0.0,0.0,NaN,996.5,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 23)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['O Xylene (µg/m³)', 'WS (m/s)', 'WD (deg)']
Dropped rows (>70% NaN): 62
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Toluene (µg/m³)        0
Xylene (µg/m³)         0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
AT (°C)                0
RH (%)                 0
RF (mm)                0
TOT-RF (mm)            0
BP (mmHg)              0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (304, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-02-19         106.63        134.22       12.03        15.58   
1  2024-02-20         106.63        147.37       11.97        15.63   
2  2024-02-21         106.63        154.05       11.93        15.61   
3  2024-02-22         106.63        139.33       11.96        15.66   
4  2024-02-23         106.63        140.21       11.97        15.65   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      26.18        31.17        18.28        2.44          28.01   
1      26.16        30.79        18.80        2.80          28.85   
2      26.14        30.92        13.06        2.57          28.20   
3      26.20        30.80        24.82        2.43          28.56   
4      26.11        31.16        14.04        2.45          28.07   

   Benzene (µg/m³)  Toluene (µg/m³)  Xylene (µg/m³)  Eth-Benzene (µg/m³)  \
0             2.59             6.98            5.78        

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),Xylene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),AT (°C),RH (%),RF (mm),TOT-RF (mm),BP (mmHg)
0,2024-02-19,0.066393,-0.603890,-1.018591,-1.458810,-1.125845,-1.104663,-0.989674,1.813200,0.644368,0.0,0.046803,0.007007,-2.820888,1.748730,-0.142390,1.421085e-14,0.0,0.0,0.0
1,2024-02-20,0.066393,-0.467635,-1.021376,-1.456919,-1.126760,-1.123268,-0.925606,2.445143,0.714033,0.0,-1.375997,-0.703043,-0.910977,-1.837199,-0.142390,1.421085e-14,0.0,0.0,0.0
2,2024-02-21,0.066393,-0.398419,-1.023234,-1.457675,-1.127674,-1.116903,-1.632819,2.041401,0.660125,0.0,-2.087397,1.427107,-1.865933,-0.940716,-0.142390,1.421085e-14,0.0,0.0,0.0
3,2024-02-22,0.066393,-0.550942,-1.021841,-1.455785,-1.124931,-1.122778,-0.183894,1.795646,0.689982,0.0,0.758202,0.717057,0.998934,-1.837199,-0.142390,1.421085e-14,0.0,0.0,0.0
4,2024-02-23,0.066393,-0.541824,-1.021376,-1.456163,-1.129046,-1.105153,-1.512075,1.830754,0.649344,0.0,-2.798796,-0.703043,0.998934,1.748730,-0.142390,1.421085e-14,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,2024-12-26,0.066393,0.859174,-0.386654,-0.403116,-0.309162,-0.152861,-0.074239,1.427012,0.631098,0.0,0.046803,0.007007,0.043978,-0.044234,-0.142390,1.421085e-14,0.0,0.0,0.0
300,2024-12-27,0.066393,0.743435,-0.613241,-0.496510,-0.551057,-0.342830,0.207907,1.251472,0.412981,0.0,0.046803,0.007007,0.043978,-0.044234,-0.142390,1.421085e-14,0.0,0.0,0.0
301,2024-12-29,0.066393,-0.654352,0.279641,0.468815,0.786456,0.654505,-0.400740,1.427012,1.024208,0.0,0.046803,0.007007,0.043978,-0.044234,5.010774,1.421085e-14,0.0,0.0,0.0
302,2024-12-30,0.066393,-0.588762,0.743030,0.204136,0.946044,1.051089,-0.691510,0.514205,0.563922,0.0,0.046803,0.007007,0.043978,-0.044234,-0.142390,1.421085e-14,0.0,0.0,0.0


In [10]:
df.to_excel('chandnichowk2024.xlsx', index=False)